In [ ]:
# ============================================
# CARREGAR OS DADOS FINAIS
# ============================================

import numpy as np
import os
import pickle
from sklearn.svm import SVC

X_final = np.load('encoded_data.npy')
y_final = np.load('outcome.npy')
final_variable_names = np.load('encoded_variables.npy', allow_pickle=True)
print(f"📂 Dados finais carregados: {X_final.shape[0]} amostras × {X_final.shape[1]} variáveis.")

# ============================================
# VERIFICAÇÃO E CORREÇÃO DE NaN — ANTES DO SVM
# ============================================

print(f"🔍 Verificando NaN em X_final: {np.isnan(X_final).sum()} valores faltantes")

if np.isnan(X_final).any():
    print("⚠️ Existem NaNs — substituindo pela média da coluna...")
    col_means = np.nanmean(X_final, axis=0)
    inds = np.where(np.isnan(X_final))
    X_final[inds] = np.take(col_means, inds[1])
    print(f"✅ Substituição concluída. NaNs restantes: {np.isnan(X_final).sum()}")
else:
    print("✅ Nenhum NaN detectado — pronto para o treino.")

# ============================================
# AVALIAÇÃO DO MODELO SVM (Linear)
# ============================================

svm_params = {'C': [0.1, 1, 10]}
model = SVC(probability=True, kernel='linear', class_weight='balanced', random_state=42)

print("\n🚀 Iniciando avaliação SVM (Linear) com GridSearchCV e Bootstrap por fold...")
svm_auc, svm_metrics, svm_models, svm_err, svm_feat, svm_probs, svm_preds = run_k_fold(
    model, 'SVM', svm_params, X_final, y_final, n_splits=5, n_bootstrap_iter=1000
)

# ============================================
# RESULTADOS MÉDIOS
# ============================================

svm_mean = np.mean(svm_metrics, axis=0)
print("\n📊 MÉDIAS — SVM (5-Fold + Bootstrap 1000):")
print(f"  Accuracy:     {svm_mean[0]:.3f}")
print(f"  Sensitivity:  {svm_mean[1]:.3f}")
print(f"  Specificity:  {svm_mean[2]:.3f}")
print(f"  PPV:          {svm_mean[7]:.3f}")
print(f"  NPV:          {svm_mean[8]:.3f}")
print(f"  AUC (média):  {np.mean(svm_auc):.3f}")

# ============================================
# SALVAR RESULTADOS
# ============================================

os.makedirs('5fold', exist_ok=True)
with open('5fold/svm_results.pkl', 'wb') as f:
    pickle.dump(
        [svm_auc, svm_metrics, svm_models, svm_err, svm_feat, svm_probs, svm_preds, final_variable_names],
        f
    )

print("\n💾 Resultados salvos com sucesso em: '5fold/svm_results.pkl'")
print("✅ Execução SVM Linear concluída.")


In [ ]:

if not isinstance(X, pd.DataFrame):
    X = pd.DataFrame(X)

X_imputed = X.copy()
skipped_cols = []

def sample_impute(series):
    """Imputa valores ausentes amostrando de valores não nulos (James, 2019)."""
    non_null_values = series.dropna().values
    n_missing = series.isna().sum()
    if n_missing > 0 and len(non_null_values) > 0:
        imputed_values = np.random.choice(non_null_values, size=n_missing, replace=True)
        series = series.copy()
        series.loc[series.isna()] = imputed_values
    return series

for col in tqdm(X_imputed.columns, desc="🔄 Imputando valores faltantes (estilo James)"):
    try:
        X_imputed[col] = sample_impute(X_imputed[col])
    except Exception as e:
        skipped_cols.append((col, str(e)))

if skipped_cols:
    print(f"⚠️ Colunas ignoradas ({len(skipped_cols)}): {[c[0] for c in skipped_cols]}")

print(f"\n✅ Dataset imputado: {X_imputed.shape} | NaN restantes: {X_imputed.isna().sum().sum()}")


In [ ]:

assert 'X_imputed' in globals() and isinstance(X_imputed, pd.DataFrame), "⚠️ Defina X_imputed antes (dataset imputado)."

def infer_type_safe(s: pd.Series) -> str:
    """Classifica variável como Binary, Continuous ou Categorical (tolerante a colunas com objetos/listas)."""
    try:
        if s.apply(lambda x: isinstance(x, (list, np.ndarray, pd.Series))).any():
            return "Invalid"
        
        s2 = s.dropna()
        nunique = s2.nunique(dropna=True)
        
        if nunique == 2:
            return 'Binary'
        if pd.api.types.is_numeric_dtype(s):
            return 'Continuous' if nunique > 10 else 'Categorical'
        if pd.api.types.is_string_dtype(s) or str(s.dtype) in ('object', 'category'):
            return 'Categorical'
        return 'Unknown'
    except Exception as e:
        return f"Error: {e}"

cols = list(X_imputed.columns)
var_types = [infer_type_safe(X_imputed[c]) for c in cols]

var_types_df = pd.DataFrame({'VAR_NAME': cols, 'VAR_TYPE': var_types})

print("Tipagem inferida (amostra):")
display(var_types_df.head(15))

print("\n Contagem de tipos:")
print(var_types_df['VAR_TYPE'].value_counts())

variables_for_dummy = var_types_df['VAR_NAME'].values
var_types_for_dummy = var_types_df['VAR_TYPE'].values

with open('ImpVars_dummy.pkl', 'wb') as f:
    pickle.dump([None, None, None, variables_for_dummy, None, var_types_for_dummy, None], f)

print(f"\n'ImpVars_dummy.pkl' salvo. Total de variáveis: {len(variables_for_dummy)}")

invalid_cols = var_types_df[var_types_df['VAR_TYPE'] == 'Invalid']
if not invalid_cols.empty:
    print(f"\n{len(invalid_cols)} colunas removidas por conter listas/arrays:")
    display(invalid_cols)


In [ ]:
#salvar os dados
X_values_imputed = X_imputed.values
columns_imputed = X_imputed.columns.values
outcome_values = y.values  

with open('processed_data.pkl', 'wb') as f:
    pickle.dump([X_values_imputed, columns_imputed, outcome_values], f)
print("💾 Arquivo 'processed_data.pkl' salvo com sucesso.")

participant_ids = df_final['NACCID'].astype(str).values
np.save('NACCID.npy', participant_ids)
print("💾 Arquivo 'NACCID.npy' salvo com sucesso.")

print(f"Shape dos dados (X) salvos: {X_values_imputed.shape}")
print(f"Número de colunas salvas: {len(columns_imputed)}")
print(f"Shape do outcome (y) salvo: {outcome_values.shape}")
print(f"Número de IDs salvos: {len(participant_ids)}")